In [ ]:
import itertools, random, statistics
from collections import deque, defaultdict
from dataclasses import dataclass, replace
from typing import Any, Dict, List, Optional, Tuple

SEED = 2026
print("Setup OK - standard library only (no API keys, no internet needed).")

In [ ]:
PERFORMATIVES = {"REQUEST", "INFORM", "PROPOSE", "ACCEPT", "REJECT", "FAILURE"}

def _brief(v):
    if isinstance(v, (list, tuple, set)):
        return f"[{len(v)} items]"
    if isinstance(v, dict):
        return "{...}"
    s = str(v).replace("\n", " ")
    return s if len(s) <= 28 else s[:27] + "..."

@dataclass
class Message:
    id: int
    sender: str
    receiver: str
    performative: str
    content: Dict[str, Any]
    reply_to: Optional[int] = None

    def short(self, width=72):
        kind = self.content.get("kind", "")
        rest = {k: v for k, v in self.content.items() if k != "kind"}
        body = ", ".join(f"{k}={_brief(v)}" for k, v in rest.items())
        text = f"{kind}: {body}" if body else kind
        return text if len(text) <= width else text[: width - 3] + "..."


class MessageBus:
    """Delivers messages one at a time and keeps a full log (the 'trace')."""

    def __init__(self, verbose=True, max_messages=300):
        self.verbose = verbose
        self.max_messages = max_messages
        self.agents: Dict[str, "Agent"] = {}
        self.queue = deque()
        self.log: List[Message] = []
        self._ids = itertools.count(1)

    def register(self, *agents):
        for a in agents:
            self.agents[a.name] = a
            a.bus = self
        return self

    def send(self, sender, receiver, performative, reply_to=None, **content):
        assert performative in PERFORMATIVES, f"unknown performative {performative}"
        if len(self.log) >= self.max_messages:
            raise RuntimeError(
                f"Message budget ({self.max_messages}) exhausted - possible livelock / runaway loop")
        msg = Message(next(self._ids), sender, receiver, performative, content, reply_to)
        self.queue.append(msg)
        self.log.append(msg)
        if self.verbose:
            print(f"#{msg.id:02d} {sender:>12} -> {receiver:<12} [{performative:<7}] {msg.short()}")
        return msg

    def run(self):
        """Process messages until the queue is empty."""
        while self.queue:
            msg = self.queue.popleft()
            self.agents[msg.receiver].handle(msg)


class Agent:
    """Base class: a name, a role, a goal, and a handle() method. Nothing else is shared."""

    def __init__(self, name, role, goal):
        self.name, self.role, self.goal = name, role, goal
        self.bus: Optional[MessageBus] = None

    def send(self, to, performative, reply_to=None, **content):
        return self.bus.send(self.name, to, performative, reply_to, **content)

    def handle(self, msg: Message):
        raise NotImplementedError

    def __repr__(self):
        return f"<{self.name}: {self.role}>"

print("Message, MessageBus and Agent defined.")

In [ ]:
@dataclass(frozen=True)
class Fact:
    id: str
    topic: str
    subtopic: str
    template: str
    value: float
    unit: str
    source: str
    derive: Optional[Tuple[str, float]] = None   # (sentence template, multiplier) for a derived claim

    @property
    def text(self):
        return self.template.format(value=f"{self.value:g}", unit=self.unit)

    def derived_text(self):
        if not self.derive:
            return None
        tpl, k = self.derive
        return tpl.format(derived=f"{self.value * k:,.0f}")


_RAW = [
    # id, topic, subtopic, sentence template, value, unit, optional derived claim
    ("sol-cost",  "solar energy", "cost",       "Utility-scale solar electricity cost has fallen about {value}{unit} since 2010.", 85, "%", None),
    ("sol-eff",   "solar energy", "efficiency", "Typical commercial silicon panels convert roughly {value}{unit} of sunlight into electricity.", 21, "%", None),
    ("sol-share", "solar energy", "adoption",   "Solar now supplies about {value}{unit} of global electricity.", 6, "%", None),
    ("sol-life",  "solar energy", "durability", "Panels are usually warrantied for about {value} {unit}.", 25, "years", None),
    ("ev-range",  "electric vehicles", "range",    "A typical new EV offers roughly {value} {unit} of range per charge.", 400, "km", None),
    ("ev-charge", "electric vehicles", "charging", "A DC fast charger adds 10-80% charge in about {value} {unit}.", 30, "minutes", None),
    ("ev-batt",   "electric vehicles", "cost",     "Battery pack prices are near {value} {unit}.", 130, "USD/kWh",
        ("A 60 kWh pack would therefore cost about {derived} USD.", 60)),
    ("ev-share",  "electric vehicles", "adoption", "EVs make up about {value}{unit} of new car sales worldwide.", 18, "%", None),
]

def build_kb(source, jitter=0.0, seed=0):
    rng = random.Random(seed)
    kb = defaultdict(lambda: defaultdict(list))
    for fid, topic, sub, tpl, val, unit, der in _RAW:
        v = val * (1 + rng.uniform(-jitter, jitter))
        kb[topic][sub].append(Fact(fid, topic, sub, tpl, round(v, 2), unit, source, der))
    return kb

KB_A = build_kb("Source-A")                          # Researcher's source = ground truth
KB_B = build_kb("Source-B", jitter=0.03, seed=7)     # independent second source (small noise)
TRUTH = {f.id: f.value for subs in KB_A.values() for fs in subs.values() for f in fs}

def cross_check(fact, kb_other, tolerance=0.10):
    """Tool used by the Verifier: does an independent source agree within a tolerance?"""
    ref = next((f for f in kb_other[fact.topic][fact.subtopic] if f.id == fact.id), None)
    return ref is not None and abs(fact.value - ref.value) <= tolerance * abs(ref.value)

print("Topics available:", list(KB_A.keys()))
print("Facts in KB:", len(TRUTH))

In [ ]:
class Planner(Agent):
    def __init__(self):
        super().__init__("Planner", "Planner", "Break a topic into researchable subtopics")

    def handle(self, msg):
        if msg.content["kind"] != "plan_request":
            return
        topic = msg.content["topic"]
        subtopics = sorted(KB_A.get(topic, {}).keys())
        if not subtopics:
            self.send(msg.sender, "FAILURE", reply_to=msg.id, kind="plan_failed",
                      reason=f"unknown topic '{topic}'")
        else:
            self.send(msg.sender, "INFORM", reply_to=msg.id, kind="plan",
                      topic=topic, subtopics=subtopics)


class Researcher(Agent):
    """Retrieves facts. Can be made unreliable to simulate hallucination / a faulty tool."""

    def __init__(self, name="Researcher", fault_rate=0.0, seed=0, force_fault=()):
        super().__init__(name, "Researcher", "Retrieve cited facts for one subtopic")
        self.fault_rate = fault_rate
        self.rng = random.Random(seed)
        self.force_fault = set(force_fault)   # subtopics that ALWAYS fail on the first attempt

    def handle(self, msg):
        if msg.content["kind"] != "research_request":
            return
        topic, sub = msg.content["topic"], msg.content["subtopic"]
        attempt = msg.content.get("attempt", 1)
        facts = list(KB_A[topic][sub])
        forced = sub in self.force_fault and attempt == 1
        if facts and (forced or self.rng.random() < self.fault_rate):
            i = self.rng.randrange(len(facts))
            facts[i] = replace(facts[i], value=facts[i].value * 10)   # decimal-slip "hallucination"
        self.send(msg.sender, "INFORM", reply_to=msg.id, kind="facts",
                  subtopic=sub, facts=facts, attempt=attempt)


class Verifier(Agent):
    def __init__(self, tolerance=0.10):
        super().__init__("Verifier", "Fact-checker", "Cross-check facts against an independent source")
        self.tolerance = tolerance

    def handle(self, msg):
        if msg.content["kind"] != "verify_request":
            return
        facts = msg.content["facts"]
        bad = [f.id for f in facts if not cross_check(f, KB_B, self.tolerance)]
        self.send(msg.sender, "INFORM", reply_to=msg.id, kind="verdict",
                  subtopic=msg.content["subtopic"], ok=not bad, bad_ids=bad)


class Writer(Agent):
    def __init__(self):
        super().__init__("Writer", "Writer", "Turn facts into a short brief")

    def handle(self, msg):
        if msg.content["kind"] != "draft_request":
            return
        c = msg.content
        facts, codes = c["facts"], set(c["codes"])
        lines = [f"# Brief: {c['topic'].title()}"]
        for f in facts:
            cite = f" [{f.source}]" if "ADD_SOURCES" in codes else ""
            lines.append(f"- {f.text}{cite}")
            d = f.derived_text()
            if d:
                lines.append(f"    => {d}")
        if c["excluded"]:
            lines.append(f"Note: excluded unverified items: {', '.join(c['excluded'])}")
        if "ADD_CONCLUSION" in codes:
            keys = "; ".join(f"{f.id}={f.value:g} {f.unit}" for f in facts)
            lines.append(f"Conclusion: {len(facts)} facts reviewed. Key figures: {keys}.")
        self.send(msg.sender, "INFORM", reply_to=msg.id, kind="draft",
                  draft="\n".join(lines), version=c["version"])


class Reviewer(Agent):
    """Checks FORM (citations, conclusion), not TRUTH. Remember this for the error-propagation part."""

    def __init__(self, always_reject=False):
        super().__init__("Reviewer", "Reviewer", "Check the draft against the style checklist")
        self.always_reject = always_reject     # a never-satisfied reviewer, used for the livelock demo

    def handle(self, msg):
        if msg.content["kind"] != "review_request":
            return
        draft, codes = msg.content["draft"], []
        if any(l.startswith("- ") and "[" not in l for l in draft.splitlines()):
            codes.append("ADD_SOURCES")
        if "Conclusion:" not in draft:
            codes.append("ADD_CONCLUSION")
        if self.always_reject:
            codes.append("MORE_DETAIL")
        self.send(msg.sender, "INFORM", reply_to=msg.id, kind="review",
                  approved=not codes, codes=codes)

print("Planner, Researcher, Verifier, Writer, Reviewer defined.")

In [ ]:
class Orchestrator(Agent):
    def __init__(self, validate=True, max_retries=2, max_review_rounds=3):
        super().__init__("Orchestrator", "Coordinator",
                         "Decompose the goal, route work, enforce quality gates")
        self.validate = validate
        self.max_retries = max_retries
        self.max_review_rounds = max_review_rounds
        self.reset()

    def reset(self):
        self.topic, self.order = None, []
        self.pending, self.facts, self._cache = set(), {}, {}
        self.retries = defaultdict(int)
        self.total_retries = 0
        self.excluded, self.codes = [], set()
        self.rounds, self._draft = 0, ""
        self.done, self.result = False, {}

    # ---- helpers -------------------------------------------------------
    def all_facts(self):
        return [f for s in self.order for f in self.facts.get(s, [])]

    def start(self, topic):
        self.reset()
        self.topic = topic
        self.send("Planner", "REQUEST", kind="plan_request", topic=topic)

    def _accept(self, sub, facts):
        self.facts[sub] = facts
        self.pending.discard(sub)
        if not self.pending:
            self._request_draft()

    def _request_draft(self):
        self.send("Writer", "REQUEST", kind="draft_request", topic=self.topic,
                  facts=self.all_facts(), codes=sorted(self.codes),
                  excluded=self.excluded, version=self.rounds + 1)

    def _finish(self, approved, escalated=False, reason=""):
        self.done = True
        facts = self.all_facts()
        self.result = dict(
            report=self._draft, approved=approved, escalated=escalated, rounds=self.rounds,
            retries=self.total_retries, excluded=list(self.excluded), reason=reason,
            n_facts=len(facts),
            # ground-truth audit: an oracle OUTSIDE the system, used only for evaluation
            wrong_facts=[f.id for f in facts if abs(f.value - TRUTH[f.id]) > 1e-9],
        )

    # ---- message handling (the protocol) --------------------------------
    def handle(self, msg):
        c, k = msg.content, msg.content["kind"]

        if k == "plan":
            self.order = list(c["subtopics"])
            self.pending = set(self.order)
            for s in self.order:
                self.send("Researcher", "REQUEST", kind="research_request",
                          topic=self.topic, subtopic=s, attempt=1)

        elif k == "plan_failed":
            self._finish(False, reason=c["reason"])

        elif k == "facts":
            sub = c["subtopic"]
            if self.validate:
                self._cache[sub] = c["facts"]
                self.send("Verifier", "REQUEST", kind="verify_request", subtopic=sub, facts=c["facts"])
            else:
                self._accept(sub, c["facts"])           # naive: trust the Researcher

        elif k == "verdict":
            sub, facts = c["subtopic"], self._cache[c["subtopic"]]
            if c["ok"]:
                self._accept(sub, facts)
            else:
                self.retries[sub] += 1
                if self.retries[sub] <= self.max_retries:
                    self.total_retries += 1
                    self.send("Researcher", "REQUEST", kind="research_request",
                              topic=self.topic, subtopic=sub, attempt=self.retries[sub] + 1,
                              feedback=f"failed cross-check: {c['bad_ids']}")
                else:                                    # give up: QUARANTINE the bad facts
                    self.excluded.extend(c["bad_ids"])
                    self._accept(sub, [f for f in facts if f.id not in c["bad_ids"]])

        elif k == "draft":
            self._draft = c["draft"]
            self.send("Reviewer", "REQUEST", kind="review_request", draft=c["draft"])

        elif k == "review":
            self.rounds += 1
            if c["approved"]:
                self._finish(True)
            elif self.max_review_rounds is not None and self.rounds >= self.max_review_rounds:
                self._finish(False, escalated=True, reason="review round limit reached")
            else:
                self.codes |= set(c["codes"])            # negotiation: carry feedback to the Writer
                self._request_draft()

print("Orchestrator defined.")

In [ ]:
def build_crew(validate=True, fault_rate=0.0, seed=0, max_retries=2, max_review_rounds=3,
               verbose=False, reviewer_always_reject=False, max_messages=300, force_fault=()):
    """Assemble bus + agents. Returns (bus, orchestrator)."""
    bus = MessageBus(verbose=verbose, max_messages=max_messages)
    orch = Orchestrator(validate=validate, max_retries=max_retries, max_review_rounds=max_review_rounds)
    bus.register(orch, Planner(), Researcher(fault_rate=fault_rate, seed=seed, force_fault=force_fault),
                 Verifier(), Writer(), Reviewer(always_reject=reviewer_always_reject))
    return bus, orch

def run_task(topic, **kw):
    bus, orch = build_crew(**kw)
    orch.start(topic)
    bus.run()
    return bus, orch

def show(orch, title):
    r = orch.result
    print("=" * 74); print(title); print("=" * 74)
    print(r["report"]); print("-" * 74)
    print(f"approved by Reviewer : {r['approved']}  (review rounds={r['rounds']}, escalated={r['escalated']})")
    print(f"ground-truth audit   : wrong facts in final report = {r['wrong_facts'] or 'none'}")
    print(f"retries={r['retries']}   quarantined={r['excluded'] or 'none'}")

print("Helpers ready.")

In [ ]:
bus, orch = run_task("solar energy", validate=True, verbose=True)
print()
show(orch, "RESULT - happy path")
print(f"\nTotal messages exchanged: {len(bus.log)}")

In [ ]:
class Bidder(Agent):
    def __init__(self, name, confidence, cost):
        super().__init__(name, "Researcher (bidder)", "Bid for research work")
        self.confidence, self.cost = confidence, cost

    def handle(self, msg):
        k = msg.content["kind"]
        if k == "cfp":                                   # call for proposals
            self.send(msg.sender, "PROPOSE", reply_to=msg.id, kind="bid",
                      confidence=self.confidence, cost=self.cost)
        elif k == "award":                               # my bid was accepted: do the work
            t, s = msg.content["topic"], msg.content["subtopic"]
            self.send(msg.sender, "INFORM", reply_to=msg.id, kind="result",
                      worker=self.name, facts=list(KB_A[t][s]))
        # kind == "rejected": nothing to do


class Manager(Agent):
    def __init__(self, bidder_names, cost_weight):
        super().__init__("Manager", "Contract manager", "Award work to the best bid")
        self.bidders, self.cost_weight = bidder_names, cost_weight

    def announce(self, topic, subtopic):
        self.topic, self.sub = topic, subtopic
        self.bids, self.winner, self.result, self.utilities = {}, None, None, {}
        for b in self.bidders:
            self.send(b, "REQUEST", kind="cfp", topic=topic, subtopic=subtopic)

    def handle(self, msg):
        k = msg.content["kind"]
        if k == "bid":
            self.bids[msg.sender] = msg.content
            if len(self.bids) == len(self.bidders):      # all bids are in
                self._award()
        elif k == "result":
            self.result = msg.content

    def _award(self):
        self.utilities = {n: b["confidence"] - self.cost_weight * b["cost"] for n, b in self.bids.items()}
        self.winner = max(self.utilities, key=self.utilities.get)
        for n in self.bidders:
            if n == self.winner:
                self.send(n, "ACCEPT", kind="award", topic=self.topic, subtopic=self.sub)
            else:
                self.send(n, "REJECT", kind="rejected", reason="lower utility")


def run_contract_net(cost_weight, label):
    print(f"\n--- {label}  (cost_weight={cost_weight}) ---")
    bus = MessageBus(verbose=True)
    bidders = [Bidder("FastBot", 0.70, 5), Bidder("CarefulBot", 0.95, 20)]
    mgr = Manager([b.name for b in bidders], cost_weight)
    bus.register(mgr, *bidders)
    mgr.announce("electric vehicles", "cost")
    bus.run()
    print("utilities:", {n: round(u, 3) for n, u in mgr.utilities.items()}, "-> winner:", mgr.winner)

run_contract_net(0.02,  "Budget-conscious manager")
run_contract_net(0.005, "Quality-critical manager")

In [ ]:
bus, orch = run_task("electric vehicles", validate=False, force_fault={"cost"})
show(orch, "NAIVE PIPELINE (no verification gate): error propagates")

In [ ]:
bus, orch = run_task("electric vehicles", validate=True, force_fault={"cost"}, verbose=True)
print()
show(orch, "GATED PIPELINE (verify -> retry -> quarantine): error contained")

In [ ]:
EXPERIMENT_TOPIC = "electric vehicles"
N_TRIALS = 300
RATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

def trial(seed, validate, fault_rate):
    bus, orch = run_task(EXPERIMENT_TOPIC, validate=validate, fault_rate=fault_rate, seed=seed)
    r = orch.result
    return dict(wrong=len(r["wrong_facts"]), msgs=len(bus.log), retries=r["retries"], excl=len(r["excluded"]))

rows = []
for p in RATES:
    naive = [trial(10_000 + i, False, p) for i in range(N_TRIALS)]
    gated = [trial(10_000 + i, True,  p) for i in range(N_TRIALS)]
    rows.append(dict(
        p=p,
        naive_err=100 * sum(r["wrong"] > 0 for r in naive) / N_TRIALS,
        gated_err=100 * sum(r["wrong"] > 0 for r in gated) / N_TRIALS,
        naive_msgs=statistics.mean(r["msgs"] for r in naive),
        gated_msgs=statistics.mean(r["msgs"] for r in gated),
        gated_excl=statistics.mean(r["excl"] for r in gated),
    ))

print(f"{'fault':>6} | {'wrong reports %':^21} | {'avg messages':^17} | quarantined facts")
print(f"{'rate':>6} | {'no gate':>9} {'gate':>10} | {'no gate':>8} {'gate':>8} | (per report, gate)")
print("-" * 74)
for r in rows:
    print(f"{r['p']:>6.1f} | {r['naive_err']:>8.1f}% {r['gated_err']:>9.1f}% | "
          f"{r['naive_msgs']:>8.1f} {r['gated_msgs']:>8.1f} | {r['gated_excl']:>8.2f}")

In [ ]:
# Chart (matplotlib if installed, otherwise a text bar chart)
try:
    import matplotlib.pyplot as plt
    xs = [r["p"] for r in rows]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(xs, [r["naive_err"] for r in rows], "o-", label="no gate")
    ax[0].plot(xs, [r["gated_err"] for r in rows], "s-", label="verification gate")
    ax[0].set_xlabel("Researcher fault rate"); ax[0].set_ylabel("% reports with a wrong fact")
    ax[0].set_title("Error propagation"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(xs, [r["naive_msgs"] for r in rows], "o-", label="no gate")
    ax[1].plot(xs, [r["gated_msgs"] for r in rows], "s-", label="verification gate")
    ax[1].set_xlabel("Researcher fault rate"); ax[1].set_ylabel("average messages per task")
    ax[1].set_title("Cost of the safety"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed - text chart of '% wrong reports':\n")
    for r in rows:
        print(f"p={r['p']:.1f}  no gate |{'#' * int(r['naive_err'] / 2):<50}| {r['naive_err']:5.1f}%")
        print(f"       gate    |{'#' * int(r['gated_err'] / 2):<50}| {r['gated_err']:5.1f}%")

In [ ]:
class PeerAgent(Agent):
    """Produces a deliverable for `send_to` once it has received one from `needs` (None = can start at once)."""

    def __init__(self, name, needs=None, send_to=None, priority=0):
        super().__init__(name, "Peer", "Produce a deliverable once its dependency is met")
        self.needs, self.send_to, self.priority = needs, send_to, priority
        self.status, self.waiting_for = "IDLE", None

    def kick(self):
        if self.needs is None:
            self._produce()
        else:
            self.status, self.waiting_for = "WAITING", self.needs

    def _produce(self, provisional=False):
        self.status, self.waiting_for = "DONE", None
        if self.send_to:
            self.send(self.send_to, "INFORM", kind="deliverable", provisional=provisional,
                      item=f"{self.name}-output")

    def handle(self, msg):
        k = msg.content["kind"]
        if self.status != "WAITING":
            return                                           # already done: ignore late messages
        if k == "fallback":                                  # guard told us to proceed anyway
            self._produce(provisional=True)
        elif k == "deliverable" and msg.sender == self.waiting_for:
            self._produce()


def find_cycle(wait_for):
    """wait_for: {agent: agent it waits for}. Returns the list of agents in a cycle, or None."""
    for start in wait_for:
        seen, cur = [], start
        while cur in wait_for and cur not in seen:
            seen.append(cur)
            cur = wait_for[cur]
        if cur in seen:
            return seen[seen.index(cur):]
    return None


def run_with_guard(agents, use_guard=True, max_interventions=5):
    bus = MessageBus(verbose=True)
    bus.register(*agents)
    for a in agents:
        a.kick()
    bus.run()
    interventions = 0
    while True:
        waiting = {a.name: a.waiting_for for a in agents if a.status == "WAITING"}
        if not waiting:
            print("  -> all agents finished.\n")
            return True
        cycle = find_cycle(waiting)
        print(f"  STALL: queue is empty but {sorted(waiting)} are still WAITING")
        print(f"  wait-for graph: {waiting}")
        if cycle:
            print("  DEADLOCK cycle:", " -> ".join(cycle + [cycle[0]]))
        if not use_guard or not cycle or interventions >= max_interventions:
            print("  -> no guard: the system hangs here.\n")
            return False
        victim = max((bus.agents[n] for n in cycle), key=lambda a: a.priority)
        print(f"  GUARD: '{victim.name}' has the highest priority in the cycle and proceeds "
              f"with provisional input")
        bus.send("Guard", victim.name, "INFORM", kind="fallback")
        bus.run()
        interventions += 1


print("### A) Writer <-> Reviewer, circular wait, NO guard")
run_with_guard([PeerAgent("Writer", needs="Reviewer", send_to="Reviewer", priority=1),
                PeerAgent("Reviewer", needs="Writer", send_to="Writer", priority=2)], use_guard=False)

print("### B) Same protocol WITH a deadlock guard (detect cycle -> break it by priority)")
run_with_guard([PeerAgent("Writer", needs="Reviewer", send_to="Reviewer", priority=1),
                PeerAgent("Reviewer", needs="Writer", send_to="Writer", priority=2)])

print("### C) A longer ring: Planner -> Researcher -> Writer -> Planner, WITH guard")
run_with_guard([PeerAgent("Planner",    needs="Writer",     send_to="Researcher", priority=3),
                PeerAgent("Researcher", needs="Planner",    send_to="Writer",     priority=1),
                PeerAgent("Writer",     needs="Researcher", send_to="Planner",    priority=2)])

print("### D) Fix by DESIGN: break the circular wait - the Writer speaks first (optimistic start)")
run_with_guard([PeerAgent("Writer", needs=None, send_to="Reviewer", priority=1),
                PeerAgent("Reviewer", needs="Writer", send_to="Writer", priority=2)], use_guard=False)

In [ ]:
print("Without a round limit:")
try:
    run_task("solar energy", max_review_rounds=None, reviewer_always_reject=True, max_messages=60)
except RuntimeError as e:
    print("  ->", e)

print("\nWith max_review_rounds=3:")
bus, orch = run_task("solar energy", max_review_rounds=3, reviewer_always_reject=True)
r = orch.result
print(f"  -> stopped after {r['rounds']} rounds, escalated={r['escalated']}, "
      f"approved={r['approved']}, reason='{r['reason']}', messages={len(bus.log)}")